In [8]:
import cv2
import numpy as np
import open3d as o3d
import json
import glob
import matplotlib.pyplot as plt
import copy
import teaserpp_python
from numpy.linalg import inv
from scipy.spatial import cKDTree
import time
from PIL import Image
import torchvision
import torch
from torchvision import transforms as T 
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from numba import jit, prange
import os
import csv 
# Load MaskRCNN 

import subprocess

globT = 0
def register_two_views_teaser(A_pcd_raw,B_pcd_raw,VOXEL_SIZE):
    
    VISUALIZE = True
    A_pcd = A_pcd_raw.voxel_down_sample(voxel_size=VOXEL_SIZE)
    B_pcd = B_pcd_raw.voxel_down_sample(voxel_size=VOXEL_SIZE)
    #if VISUALIZE:
     #   o3d.visualization.draw_geometries([A_pcd,B_pcd]) # plot downsampled A and B 

    A_xyz = pcd2xyz(A_pcd) # np array of size 3 by N
    B_xyz = pcd2xyz(B_pcd) # np array of size 3 by M

    print("Extracting FPFH features")
    # extract FPFH features
    A_feats = extract_fpfh(A_pcd,VOXEL_SIZE)
    B_feats = extract_fpfh(B_pcd,VOXEL_SIZE)
    print(A_feats.shape)
    print("Computing FPFH correspondences")
    # establish correspondences by nearest neighbour search in feature space
    corrs_A, corrs_B = find_correspondences(
        A_feats, B_feats, mutual_filter=True)
    A_corr = A_xyz[:,corrs_A] # np array of size 3 by num_corrs
    B_corr = B_xyz[:,corrs_B] # np array of size 3 by num_corrs

    num_corrs = A_corr.shape[1]
    print(f'FPFH generates {num_corrs} putative correspondences.')

    # visualize the point clouds together with feature correspondenc
    # robust global registration using TEASER++
    NOISE_BOUND = VOXEL_SIZE
    teaser_solver = get_teaser_solver(NOISE_BOUND)
    teaser_solver.solve(A_corr,B_corr)
    solution = teaser_solver.getSolution()
    R_teaser = solution.rotation
    t_teaser = solution.translation
    T_teaser = Rt2T(R_teaser,t_teaser)

    # Visualize the registration results
    A_pcd_T_teaser = copy.deepcopy(A_pcd).transform(T_teaser)
    #o3d.visualization.draw_geometries([A_pcd_T_teaser,B_pcd])

    # local refinement using ICP
    icp_sol = o3d.pipelines.registration.registration_icp(
          A_pcd, B_pcd, NOISE_BOUND, T_teaser,
          o3d.pipelines.registration.TransformationEstimationPointToPoint(),
          o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100))
    T_icp = icp_sol.transformation

    # visualize the registration after ICP refinement
    A_pcd_T_icp = copy.deepcopy(A_pcd).transform(T_icp)
    if VISUALIZE:
        Acopy = copy.deepcopy(A_pcd_T_icp).paint_uniform_color([0.0,0.0,1])
        Bcopy = copy.deepcopy(B_pcd).paint_uniform_color([1.0,0.0,0.0])
        o3d.visualization.draw_geometries([Acopy,Bcopy])
    tformed_A = copy.deepcopy(A_pcd_raw).transform(T_icp)
    res = o3d.geometry.PointCloud()
    res = tformed_A + B_pcd_raw
    
    return res,T_icp

def pcd2xyz(pcd):
    return np.asarray(pcd.points).T

def extract_fpfh(pcd, voxel_size):
    radius_normal = voxel_size * 2
    pcd.estimate_normals(
      o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
      pcd, o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    return np.array(fpfh.data).T

def find_knn_cpu(feat0, feat1, knn=1, return_distance=False):
    feat1tree = cKDTree(feat1)
    dists, nn_inds = feat1tree.query(feat0, k=knn, workers=10)
    if return_distance:
        return nn_inds, dists
    else:
        return nn_inds

def find_correspondences(feats0, feats1, mutual_filter=True):
    nns01 = find_knn_cpu(feats0, feats1, knn=1, return_distance=False)
    corres01_idx0 = np.arange(len(nns01))
    corres01_idx1 = nns01

    if not mutual_filter:
        return corres01_idx0, corres01_idx1

    nns10 = find_knn_cpu(feats1, feats0, knn=1, return_distance=False)
    corres10_idx1 = np.arange(len(nns10))
    corres10_idx0 = nns10

    mutual_filter = (corres10_idx0[corres01_idx1] == corres01_idx0)
    corres_idx0 = corres01_idx0[mutual_filter]
    corres_idx1 = corres01_idx1[mutual_filter]

    return corres_idx0, corres_idx1

def get_teaser_solver(noise_bound):
    solver_params = teaserpp_python.RobustRegistrationSolver.Params()
    solver_params.cbar2 = 1.0
    solver_params.noise_bound = noise_bound
    solver_params.estimate_scaling = False
    solver_params.inlier_selection_mode = \
        teaserpp_python.RobustRegistrationSolver.INLIER_SELECTION_MODE.PMC_EXACT
    solver_params.rotation_tim_graph = \
        teaserpp_python.RobustRegistrationSolver.INLIER_GRAPH_FORMULATION.CHAIN
    solver_params.rotation_estimation_algorithm = \
        teaserpp_python.RobustRegistrationSolver.ROTATION_ESTIMATION_ALGORITHM.GNC_TLS
    solver_params.rotation_gnc_factor = 1.4
    solver_params.rotation_max_iterations = 10000
    solver_params.rotation_cost_threshold = 1e-16
    solver = teaserpp_python.RobustRegistrationSolver(solver_params)
    return solver

def Rt2T(R,t):
    T = np.identity(4)
    T[:3,:3] = R
    T[:3,3] = t
    return T 

def save_results(animal_id, data_path, mesh, pcd_downsampled, sa, v):
    # Save the mesh and PCD
    mesh_filename = os.path.join(data_path, f"{animal_id}_mesh.ply")
    pcd_filename = os.path.join(data_path, f"{animal_id}_pcd_downsampled.ply")
    
    #print(mesh_filename)
    #print(pcd_filename)
    o3d.io.write_triangle_mesh(mesh_filename, mesh)
    
    o3d.io.write_point_cloud(pcd_filename, pcd_downsampled)

    CSV_PATH = '/home/vigir3d/Datasets/cattle_scans/cow_measurements.csv'
    # Check if CSV file exists to decide whether to write headers
    write_header = not os.path.exists(CSV_PATH)
  # Save SA & V to the CSV file in append mode
    with open(CSV_PATH, 'a', newline='') as csvfile:
        fieldnames = ['Animal ID', 'SA', 'V']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        if write_header:
            writer.writeheader()

        writer.writerow({'Animal ID': animal_id, 'SA': sa, 'V': v})

        
@jit(nopython=True, parallel=True)
def numba_eliminate_flying_pixels(depth_image, ws, threshold):
    height, width = depth_image.shape
    result = np.zeros_like(depth_image, dtype=np.float64)
    
    for cy in prange(height):
        for cx in prange(width):
            x_start, x_end = max(0, cx - ws), min(width, cx + ws + 1)
            y_start, y_end = max(0, cy - ws), min(height, cy + ws + 1)
            window = depth_image[y_start:y_end, x_start:x_end]
            result[cy, cx] = np.sum(np.abs(window - depth_image[cy, cx]))
    
    return result

def colored_ICP(source, target):
    
    voxel_radius = [0.04, 0.02, 0.01]
    max_iter = [50, 30, 14]
    current_transformation = np.identity(4)
    #print("3. Colored point cloud registration")
    for scale in range(3):
        iters = max_iter[scale]
        radius = voxel_radius[scale]
        #print("iteration: ", iters, radius, scale)

        #print("3-1. Downsample with a voxel size %.2f" % radius)
        source_down = copy.deepcopy(source).voxel_down_sample(radius)
        target_down = copy.deepcopy(target).voxel_down_sample(radius)

        #print("3-2. Estimate normal.")
        source_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))
        target_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))

        #print("3-3. Applying colored point cloud registration")
        result_icp = o3d.pipelines.registration.registration_colored_icp(
            source_down, target_down, radius, current_transformation,
            o3d.pipelines.registration.TransformationEstimationForColoredICP(),
            o3d.pipelines.registration.ICPConvergenceCriteria(relative_fitness=1e-6,
                                                              relative_rmse=1e-6,
                                                              max_iteration=iters))
        current_transformation = result_icp.transformation
    
   
        #draw_registration_result(source, target, current_transformation)
    return current_transformation

def backproject_o3d(rgbd_frame, intrinsics):
    
    rgbdc = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgbd_frame.color, rgbd_frame.depth, depth_trunc=4.0, convert_rgb_to_intensity=False)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbdc, intrinsics)
    n_radius = 0.01*2.0
    pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=n_radius, max_nn=30))
    
    return pcd



def load_maskrcnn_model(model_path):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, 2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    return model

def get_model_mask(model_generic, image):
    proba_threshold = 0.5
    ig = transform(image)
    with torch.no_grad():
        prediction = model_generic([ig.to(device)])
        
    if(prediction[0]['masks'].nelement() == 0):
        XX = torch.empty((0,0), dtype=torch.int64)
        return XX
    predicted_mask = prediction[0]
    predicted_mask = predicted_mask['masks'][0] > proba_threshold
    
    predicted_mask = predicted_mask.squeeze(1)
    mask = predicted_mask.cpu().detach()
    return mask


def segment_images_modified(last_frame):
    depth_image_array = np.asarray(last_frame.depth)
    #print("Pre:",np.max(depth_image_array), " -- ", np.min(depth_image_array))
    depth_PIL = Image.fromarray(np.asarray(last_frame.depth)).convert("RGB")
    rgb_image_array = np.asarray(last_frame.color)
    rgb_PIL = Image.fromarray(rgb_image_array)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    rgb_mask = get_model_mask(model_rgb, rgb_PIL)
    depth_mask = get_model_mask(model_depth, depth_PIL)

    if depth_mask.nelement() == 0:
        mask_combined = rgb_mask
    else:
        mask_combined = depth_mask | rgb_mask  # 1 vote arbitration (OR the masks)

    # Convert tensor to numpy array and ensure the right datatype
    mask_combined = mask_combined.numpy().astype(rgb_image_array.dtype)
    mask_image = mask_combined.swapaxes(0, 2).swapaxes(0, 1)
    mask_image = (mask_image > 0).astype(rgb_image_array.dtype)   

    fg_image_rgb = rgb_image_array * mask_image

    # For the depth image:
    squeezed_mask = np.squeeze(mask_image)
    fg_image_depth = (depth_image_array * squeezed_mask)*1000 # upscale because re-inserting in "last_frame" rescales
    
    #print("Post:" ,np.max(fg_image_depth), " -- ", np.min(fg_image_depth))
    last_frame.color = o3d.geometry.Image(fg_image_rgb)
    last_frame.depth = o3d.geometry.Image(fg_image_depth)

    return last_frame




def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

def cluster_point_cloud_new(outlier_cloud):
    cloud_colors = copy.deepcopy(np.asarray(outlier_cloud.colors).T)
    
    with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Error) as cm:
        labels = np.array(outlier_cloud.cluster_dbscan(eps=0.05, min_points=10, print_progress=False))

    # Identify the largest cluster
    values, counts = np.unique(labels, return_counts=True)
    ind = np.argmax(counts)
    largest_cluster_label = values[ind]
    #print(f"Largest cluster label: {largest_cluster_label}")

    # Filter points, normals, and colors for the largest cluster
    cloud_xyz = pcd2xyz(outlier_cloud)
    cloud_normals = pcd2normals(outlier_cloud)

    cloud_filtered = cloud_xyz[:, labels == largest_cluster_label]
    normals_filtered = cloud_normals[:, labels == largest_cluster_label]
    colors_filtered = cloud_colors[:, labels == largest_cluster_label]

    # Create a point cloud for the largest cluster
    pcd_filtered_largest_cluster = o3d.geometry.PointCloud()
    pcd_filtered_largest_cluster.points = o3d.utility.Vector3dVector(cloud_filtered.T)
    pcd_filtered_largest_cluster.normals = o3d.utility.Vector3dVector(normals_filtered.T)
    pcd_filtered_largest_cluster.colors = o3d.utility.Vector3dVector(colors_filtered.T)

    #o3d.visualization.draw_geometries([pcd_filtered_largest_cluster])
    return pcd_filtered_largest_cluster

def load_calibration(tform_path, num_transforms=5):
    transforms = [np.eye(4)]  # Initialize with identity matrix
    
    # Load transformations from file
    for i in range(1, num_transforms+1):
        filename = tform_path + f"H_0_{i}.txt"
        
        if not os.path.exists(filename):
            raise FileNotFoundError(f"The file {filename} does not exist!")
        
        transforms.append(np.loadtxt(filename))
    
    return transforms

def upsample_using_reference_normals_new(sparse_pcd, dense_pcd, search_radius=0.02, angle_threshold=30):
    # Ensure the point clouds have normals
    if not sparse_pcd.has_normals():
        sparse_pcd.estimate_normals()
    if not dense_pcd.has_normals():
        dense_pcd.estimate_normals()

    dense_points = np.asarray(dense_pcd.points)
    dense_tree = cKDTree(dense_points)

    sparse_points = np.asarray(sparse_pcd.points)
    sparse_normals = np.asarray(sparse_pcd.normals)
    sparse_colors = np.asarray(sparse_pcd.colors)

    
    # This retrieves the indices of all neighbors within the search_radius
    neighbor_indices = dense_tree.query_ball_point(sparse_points, search_radius, workers=-1)
    
    valid_indices = []
    for i, idx_row in enumerate(neighbor_indices):
        neighbors = dense_points[idx_row]
        neighbor_normals = np.asarray(dense_pcd.normals)[idx_row]
        neighbor_colors = np.asarray(dense_pcd.colors)[idx_row]

        # Calculate angles between sparse point's normal and all its neighbors' normals
        angles = np.degrees(np.arccos(np.clip(np.dot(sparse_normals[i], neighbor_normals.T), -1.0, 1.0)))

        # Filtering neighbors based on angle threshold
        valid_neighbor_indices = np.array(idx_row)[angles < angle_threshold]
        valid_indices.extend(valid_neighbor_indices)

    unique_valid_indices = np.unique(valid_indices)
    final_valid_neighbors = dense_points[unique_valid_indices]
    final_valid_normals = np.asarray(dense_pcd.normals)[unique_valid_indices]
    final_valid_colors = np.asarray(dense_pcd.colors)[unique_valid_indices]
    
    upsampled_points = np.vstack([sparse_points, final_valid_neighbors])
    upsampled_normals = np.vstack([sparse_normals, final_valid_normals])
    upsampled_colors = np.vstack([sparse_colors, final_valid_colors])
    
    upsampled_pcd = o3d.geometry.PointCloud()
    upsampled_pcd.points = o3d.utility.Vector3dVector(upsampled_points)
    upsampled_pcd.normals = o3d.utility.Vector3dVector(upsampled_normals)
    upsampled_pcd.colors = o3d.utility.Vector3dVector(upsampled_colors)
    
    return upsampled_pcd

def process_file_list_comp(file_id, dpath,tform):
    global globT
    ws = 2
    # Format filenames
    dpath = dpath.rstrip('/')

    # Split by the directory separator and get the last part
    suffix = os.path.basename(dpath)
    
    file_suffix = suffix + "_nano_" + str(file_id)
    # Construct full file paths using os.path.join for better cross-platform compatibility
    depth_file = os.path.join(dpath, file_suffix + "_depth.png")
    color_file = os.path.join(dpath, file_suffix + ".jpg")
    intrinsics_file = os.path.join(dpath, file_suffix + "_intrinsics.txt")
    # Read the files
    depth = o3d.io.read_image(depth_file)
    color = o3d.io.read_image(color_file)
    #depth = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11_depth.png')
    #color = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11.jpg')
    K = np.loadtxt(intrinsics_file)
      

    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic()
    camera_intrinsics.set_intrinsics(int(K[0]), int(K[1]), K[2], K[3], K[4], K[5])

    rgbdc = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, depth, depth_trunc=4.0, convert_rgb_to_intensity=False
    )

    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=4.0 / 512.0, sdf_trunc=0.04, color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
    )
    
    volume.integrate(rgbdc, camera_intrinsics, np.eye(4))
    pcd_tsdf = volume.extract_point_cloud()
    pcd_tsdf.transform(tform)
    depth_np = np.asarray(rgbdc.depth)
    
    if ( depth_np.max() > 20 ) : # Depth is in millimeters
        flying_pixel_filter_threshold = 100 # 100
    else : # meters
        flying_pixel_filter_threshold = 0.1 # 0.1
        #print(flying_pixel_filter_threshold, " -- ", depth_np.max())

    result_mask = numba_eliminate_flying_pixels(depth_np.copy(), ws, flying_pixel_filter_threshold)
    depth_np[result_mask > flying_pixel_filter_threshold] = 0
    # Re-insert filtered depth into the rgbd image
    rgbdc.depth = o3d.geometry.Image(depth_np)
    # Apply maskrcnn to filtered depth image
    t1 = time.time()
    masked_rgbd = segment_images_modified(rgbdc)
    t2 = time.time()
    T = t2-t1
    globT+=T
    #print("Duration for maskrcnn 1 image ",file_id, ": ", T , "s")
    # necessary lol
    rgbdc_new = o3d.geometry.RGBDImage.create_from_color_and_depth(
                    masked_rgbd.color, masked_rgbd.depth, depth_trunc=4.0, convert_rgb_to_intensity=False)
    d = np.asarray(rgbdc_new.depth)

    v = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=4.0 / 512.0, sdf_trunc=0.04, color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
    )

    v.integrate(rgbdc_new, camera_intrinsics, np.eye(4))
    pcd_tmp = v.extract_point_cloud()
    pcd_tmp.transform(tform)
    #pcd_tmp = o3d.geometry.PointCloud.create_from_rgbd_image(rgbdc_new,
     #                                                   camera_intrinsics)

    return pcd_tmp, pcd_tsdf

def perform_pairwise_alignment(pcds_tsdf,pcds_cropped):
    """Compute and apply transformations."""
    t01 = colored_ICP(pcds_tsdf[0], pcds_tsdf[1])
    t52 = colored_ICP(pcds_tsdf[5], pcds_tsdf[2])
    t43 = colored_ICP(pcds_tsdf[4], pcds_tsdf[3])
    t12 = colored_ICP(pcds_tsdf[1], pcds_tsdf[2])
    t32 = colored_ICP(pcds_tsdf[3], pcds_tsdf[2])

    H0 = t12 @ t01
    H1 = t12
    H3 = t32
    H4 = t32 @ t43
    H5 = t52

    # Transform the point clouds
    p0 = copy.deepcopy(pcds_cropped[0]).transform(H0)
    p1 = copy.deepcopy(pcds_cropped[1]).transform(H1)
    p3 = copy.deepcopy(pcds_cropped[3]).transform(H3)
    p4 = copy.deepcopy(pcds_cropped[4]).transform(H4)
    p5 = copy.deepcopy(pcds_cropped[5]).transform(H5)
    
    d0 = copy.deepcopy(pcds_tsdf[0]).transform(H0)
    d1 = copy.deepcopy(pcds_tsdf[1]).transform(H1)
    d3 = copy.deepcopy(pcds_tsdf[3]).transform(H3)
    d4 = copy.deepcopy(pcds_tsdf[4]).transform(H4)
    d5 = copy.deepcopy(pcds_tsdf[5]).transform(H5)
    
    pcd_combined = o3d.geometry.PointCloud()
    pcd_combined = p0+p1+pcds_cropped[2]+p3+p4+p5
    
    ptsdf_combined = o3d.geometry.PointCloud()
    ptsdf_combined = d0+d1+pcds_tsdf[2]+d3+d4+d5

    return pcd_combined, ptsdf_combined
def pcd2normals(pcd):
    return np.asarray(pcd.normals).T


def call_cpp_program(directory_path):
    # Adjust this to the path of your compiled C++ executable if it's not in the system's PATH.
    cpp_executable = "/home/vigir3d/Software/programs/k4a-read-mkvs/build/k4a_read_mkv"
    command = cpp_executable + " " + directory_path + "*.mkv"
    print(command)
    # Call the C++ program.
    result = subprocess.run(command, shell=True, check=True)

    # Check if the C++ program executed without errors.
    if result.returncode != 0:
        print(f"Error running {cpp_executable}")
        return False

    return True



In [ ]:
def load_pcds(path):
    files = glob.glob(path+"*.ply")
    pcds = [o3d.io.read_point_cloud(file) for file in files]
    return pcds

In [2]:
def load_tforms_teaser():
    dpath = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_1_1/'


    r01 = np.loadtxt(dpath+"r01.txt" )
    r12 = np.loadtxt(dpath+"r12_0031.txt")
    r32 = np.loadtxt(dpath+"r32_0026.txt")
    r52 = np.loadtxt(dpath+"r52_005.txt")
    r43 = np.loadtxt(dpath+"r43_0031.txt")

    h0 = r12@r01
    h1 = r12
    h3 = r32
    h4 = r32@r43
    h5 = r52
    return [h0,h1,np.eye(4),h3,h4,h5]
#     a0 = copy.deepcopy(pcds[0]).transform(h0)
#     a1 = copy.deepcopy(pcds[1]).transform(h1)
#     a2 = copy.deepcopy(pcds[2]).transform(np.eye(4))
#     a3 = copy.deepcopy(pcds[3]).transform(h3)
#     a4 = copy.deepcopy(pcds[4]).transform(h4)
#     a5 = copy.deepcopy(pcds[5]).transform(h5)
#     return [a0,a1,a2, a3,a4,a5]

In [ ]:
path = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_11_1/'
pcds = load_pcds(path)

In [ ]:
o3d.visualization.draw_geometries(pcds)



In [28]:
def combine_pcds(pcds):
    pcd_combined = o3d.geometry.PointCloud()
    
    pcd_combined = pcds[0]+pcds[1]+pcds[2]+pcds[3]+pcds[4]+pcds[5]
    
    return pcd_combined

In [112]:




s = time.time()
rgb_model_path = '/home/vigir3d/Datasets/cattle_scans/maskrcnn_data/maskrcnn_v2.pth'
depth_model_path = '/home/vigir3d/Datasets/cattle_scans/maskrcnn_data/maskrcnn_depth_best.pth'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loads the maskrcnn trained for depth and rgb
model_rgb = load_maskrcnn_model(rgb_model_path)
model_depth = load_maskrcnn_model(depth_model_path)


transform = T.ToTensor()
# These two will need to be requested from command line
# -i   | input data path
# - t  | calibration data path 
dpath = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_18_1/'


s1 = time.time()
success = call_cpp_program(dpath)
s2 = time.time()

if not success:
    print("Failed to process MKV files with the C++ program.")
else:
    print("success!!")
    print("Load Duration: ", s2-s1, "s")
    #dpath = '/home/vigir3d/Datasets/cattle_scans/farm_scan1/Animal_482_2/'
  
    tforms = load_tforms_teaser()
    file_ids = [11,12,14,16,17,18]  # exclude head cameras

    # Generate a list of volumes using list comprehension
    start = time.time()
    
    results = [process_file_list_comp(file_id, dpath, tform) for file_id, tform in zip(file_ids,tforms)]
    pcds,ptsdf = zip(*results)
    end = time.time()

    # For now, no need to do registration .. Just transform
    r1 = time.time()
    #pcd_all, ptsdf_all = perform_pairwise_alignment(ptsdf,pcds)
    pcd_all = combine_pcds(pcds)
    ptsdf_all = combine_pcds(ptsdf)
    r2 = time.time()

    p1 = time.time()
    # cluster first to remove extra noise
    pcd_clustered = cluster_point_cloud_new(pcd_all)
    # After clustering, we upsample from the initial
    p2 = time.time()

    u1 = time.time()
    pcd_upsampled = upsample_using_reference_normals_new(pcd_clustered, ptsdf_all)
    u2 = time.time()

    tn1 = time.time()
    pcd_downsampled = pcd_upsampled.voxel_down_sample(0.01)
    #pcd_downsampled.orient_normals_towards_camera_location()
    #pcd_downsampled.orient_normals_consistent_tangent_plane(30)
    tn2 = time.time()

    m1 = time.time()
    with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Error) as cm:
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_downsampled, depth=6)

    sa = mesh.get_surface_area()
    if(mesh.is_watertight()):
        #print("Is watertight 1")
        v =  mesh.get_volume()
    else : 
        v = 0.0
    m2 = time.time()
    animal_id = os.path.basename(os.path.normpath(dpath))
    save_results(animal_id, dpath, mesh, pcd_downsampled, sa, v)

    e = time.time()
    print(f'Total time spent maskrcnn segmentation: {globT:.2f} seconds')
    globT = 0


    print("Load and Segment Duration: ", end-start, "s")

    print("Register clouds: ", r2-r1, " s")

    print("Clustering: ", p2-p1, " s")

    print("Upsampling: ", u2-u1, " s")
    print("Down Sampling: ", tn2-tn1, " s")    
    print("Meshing: ", m2-m1, " s")
    print("Total time: ", e-s, " s")


/home/vigir3d/Software/programs/k4a-read-mkvs/build/k4a_read_mkv /home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_18_1/*.mkv
success!!
Load Duration:  0.7128031253814697 s
Total time spent maskrcnn segmentation: 0.92 seconds
Load and Segment Duration:  3.491074800491333 s
Register clouds:  0.06489706039428711  s
Clustering:  0.6749579906463623  s
Upsampling:  5.449759483337402  s
Down Sampling:  0.020071029663085938  s
Meshing:  0.2219548225402832  s
Total time:  11.50997018814087  s


In [109]:
dpath

'/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_14_1/'

In [123]:
def process_file_list_comp_v2(file_id, dpath,tform): # This applies flying pixel filter to unmaskrcnn'd data
    global globT
    ws = 2
    # Format filenames
    dpath = dpath.rstrip('/')

    # Split by the directory separator and get the last part
    suffix = os.path.basename(dpath)
    
    file_suffix = suffix + "_nano_" + str(file_id)
    # Construct full file paths using os.path.join for better cross-platform compatibility
    depth_file = os.path.join(dpath, file_suffix + "_depth.png")
    color_file = os.path.join(dpath, file_suffix + ".jpg")
    intrinsics_file = os.path.join(dpath, file_suffix + "_intrinsics.txt")
    # Read the files
    depth = o3d.io.read_image(depth_file)
    color = o3d.io.read_image(color_file)
    #depth = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11_depth.png')
    #color = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11.jpg')
    K = np.loadtxt(intrinsics_file)
      

    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic()
    camera_intrinsics.set_intrinsics(int(K[0]), int(K[1]), K[2], K[3], K[4], K[5])
    
    depth_np = np.asarray(depth)
    if ( depth_np.max() > 20 ) : # Depth is in millimeters
        flying_pixel_filter_threshold = 5000 # 100
        print("Millimeters")
    else : # meters
        flying_pixel_filter_threshold = 0.1 # 0.1
        print("meters")
        #print(flying_pixel_filter_threshold, " -- ", depth_np.max())

    result_mask = numba_eliminate_flying_pixels(depth_np.copy(), ws, 0.001)
    depth_np[result_mask >600000] = 0
    # Re-insert filtered depth into the rgbd image
    

    rgbdc = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, o3d.geometry.Image(depth_np), depth_trunc=4.0, convert_rgb_to_intensity=False
    )
    #rgbdc.depth = o3d.geometry.Image(depth_np)
    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=4.0 / 512.0, sdf_trunc=0.04, color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
    )
    
    volume.integrate(rgbdc, camera_intrinsics, np.eye(4))
    pcd_tsdf = volume.extract_point_cloud()
    pcd_tsdf.transform(tform)
    
    return pcd_tsdf


# 1,13,15,8,18,21,7

ID = 7
Animal_ID = 'Animal_'+ str(ID)+"_1"
dpath = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/'+Animal_ID+'/'

pcds = [process_file_list_comp_v2(file_id, dpath, tform) for file_id, tform in zip(file_ids,tforms)]
o3d.visualization.draw_geometries(pcds)

fname = dpath + Animal_ID + '_reg.ply'
combine_write_pcds(pcds,fname)

FileNotFoundError: /home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_7_1/Animal_7_1_nano_11_intrinsics.txt not found.

In [117]:
fname

'/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_18_1/Animal_18_1_reg.ply'

[Open3D WARNING] Write Ply clamped color value to valid range


In [108]:
def combine_write_pcds(pcds,fname):
    pcd_comb = o3d.geometry.PointCloud()
    for pcd in pcds:
        pcd_comb+=pcd
    o3d.io.write_point_cloud(fname,pcd_comb)

In [15]:
o3d.visualization.draw_geometries([pcd_downsampled])

In [16]:
pcd_ds_5_no_upsamp = copy.deepcopy(pcd_downsampled)
mesh_5_no_upsamp = copy.deepcopy(mesh)

In [27]:
draw_registration_result(pcd_ds_14_no_upsamp, pcd_downsampled,np.eye(4))

In [24]:
pcd_ds_14_no_upsamp = copy.deepcopy(pcd_downsampled)
mesh_14_no_upsamp = copy.deepcopy(mesh)


In [ ]:
for p in ptsdf : 
    o3d.visualization.draw_geometries([p])

In [ ]:
o3d.visualization.draw_geometries([ptsdf_all])

In [ ]:


def upsample_using_reference_normals_new(sparse_pcd, dense_pcd, search_radius=0.02, angle_threshold=30):
    # Ensure thae point clouds have normals
    if not sparse_pcd.has_normals():
        sparse_pcd.estimate_normals()
    if not dense_pcd.has_normals():
        dense_pcd.estimate_normals()

    dense_points = np.asarray(dense_pcd.points)
    dense_tree = cKDTree(dense_points)

    sparse_points = np.asarray(sparse_pcd.points)
    sparse_normals = np.asarray(sparse_pcd.normals)
    
    # This retrieves the indices of all neighbors within the search_radius
    neighbor_indices = dense_tree.query_ball_point(sparse_points, search_radius, workers=-1)
    
    valid_indices = []
    for i, idx_row in enumerate(neighbor_indices):
        neighbors = dense_points[idx_row]
        neighbor_normals = np.asarray(dense_pcd.normals)[idx_row]

        # Calculate angles between sparse point's normal and all its neighbors' normals
        angles = np.degrees(np.arccos(np.clip(np.dot(sparse_normals[i], neighbor_normals.T), -1.0, 1.0)))

        # Filtering neighbors based on angle threshold
        valid_neighbor_indices = np.array(idx_row)[angles < angle_threshold]
        valid_indices.extend(valid_neighbor_indices)
#     valid_indices = []
#     for i, idx_row in enumerate(neighbor_indices):
#         neighbors = dense_points[idx_row]
#         neighbor_normals = np.asarray(dense_pcd.normals)[idx_row]
        
#         # Calculate angles between sparse point's normal and all its neighbors' normals
#         angles = np.degrees(np.arccos(np.clip(np.dot(sparse_normals[i], neighbor_normals.T), -1.0, 1.0)))

#         # Filtering neighbors based on angle threshold
#         valid_for_current = idx_row[angles < angle_threshold]
#         valid_indices.extend(valid_for_current)

    unique_valid_indices = np.unique(valid_indices)
    final_valid_neighbors = dense_points[unique_valid_indices]
    
    upsampled_points = np.vstack([sparse_points, final_valid_neighbors])

    upsampled_pcd = o3d.geometry.PointCloud()
    upsampled_pcd.points = o3d.utility.Vector3dVector(upsampled_points)
    upsampled_pcd.estimate_normals()

    return upsampled_pcd

In [ ]:
import subprocess
import time
import copy
import os
def call_cpp_program(directory_path):
    # Adjust this to the path of your compiled C++ executable if it's not in the system's PATH.
    cpp_executable = "/home/vigir3d/Software/programs/k4a-read-mkvs/build/k4a_read_mkv"
    command = cpp_executable + " " + directory_path + "*.mkv"
    print(command)
    # Call the C++ program.
    result = subprocess.run(command, shell=True, check=True)

    # Check if the C++ program executed without errors.
    if result.returncode != 0:
        print(f"Error running {cpp_executable}")
        return False

    return True

def apply_ICP(pcds_tsdf):
    t01 = test_ICP(pcds_tsdf[0],pcds_tsdf[1])
    t12 = test_ICP(pcds_tsdf[1],pcds_tsdf[2])
    t32 = test_ICP(pcds_tsdf[3],pcds_tsdf[2])
    t43 = test_ICP(pcds_tsdf[4],pcds_tsdf[3])
    t52 = test_ICP(pcds_tsdf[5],pcds_tsdf[2])

    h0 = t12@t01
    h1 = t12
    h3 = t32
    h4 = t32@t43
    h5 = t52

    p0 = copy.deepcopy(pcds_tsdf[0]).transform(h0)
    p1 = copy.deepcopy(pcds_tsdf[1]).transform(h1)
    p3 = copy.deepcopy(pcds_tsdf[3]).transform(h3)
    p4 = copy.deepcopy(pcds_tsdf[4]).transform(h4)
    p5 = copy.deepcopy(pcds_tsdf[5]).transform(h5)

    pcd_all = o3d.geometry.PointCloud()
    pcd_all = p0+p1+pcds_tsdf[2]+p3+p4+p5
    return pcd_all

def process_file_list_comp(file_id, dpath,tform):
    ws = 2
    flying_pixel_filter_threshold = 1000
    # Format filenames
    dpath = dpath.rstrip('/')

    # Split by the directory separator and get the last part
    suffix = os.path.basename(dpath)
    
    file_suffix = suffix + "_nano_" + str(file_id)
    # Construct full file paths using os.path.join for better cross-platform compatibility
    depth_file = os.path.join(dpath, file_suffix + "_depth.png")
    color_file = os.path.join(dpath, file_suffix + ".jpg")
    intrinsics_file = os.path.join(dpath, file_suffix + "_intrinsics.txt")
    # Read the files
    depth = o3d.io.read_image(depth_file)
    color = o3d.io.read_image(color_file)
    #depth = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11_depth.png')
    #color = o3d.io.read_image(dpath + 'Animal_box3_u_nano_11.jpg')
    K = np.loadtxt(intrinsics_file)
      
    
    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic()
    camera_intrinsics.set_intrinsics(int(K[0]), int(K[1]), K[2], K[3], K[4], K[5])

    rgbdc = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, depth, depth_trunc=4.0, convert_rgb_to_intensity=False
    )

    volume = o3d.pipelines.integration.ScalableTSDFVolume(
        voxel_length=4.0 / 512.0, sdf_trunc=0.04, color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
    )

    volume.integrate(rgbdc, camera_intrinsics, np.eye(4))
    pcd_tsdf = volume.extract_point_cloud()
    pcd_tsdf.transform(tform)
    
    return pcd_tsdf


if __name__ == "__main__":
    start = time.time()
    #for i in range(10):
    dpath = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_13_6/'
    new_dpath = copy.deepcopy(dpath).rstrip('/')

    # Split by the directory separator and get the last part
    suffix = os.path.basename(new_dpath)
    tform_path = '/home/vigir3d/Datasets/cattle_scans/farm_07_28/Animal_calib_new1/'
    tforms = load_calibration(tform_path)
    boxes = False
    if boxes: 
        file_ids = range(11,17)
    else:
        file_ids = [11,12,14,16,17,18]  # exclude head cameras

    # Generate a list of volumes using list comprehension

    #success = call_cpp_program(dpath)
    success = True
    if success: 
        pcds_tsdf_13 = [process_file_list_comp(file_id, dpath,tform) for file_id,tform in zip(file_ids,tforms)]
        #pcd_tmp = apply_ICP(pcds_tsdf)
        #pcds_all = perform_pairwise_alignment(pcds_tsdf)
        #pcds_combined = o3d.geometry.PointCloud()
        #pcds_combined = pcds_tsdf[0]+pcds_tsdf[1]+pcds_tsdf[2]+pcds_tsdf[3]+pcds_tsdf[4]+pcds_tsdf[5]
        #pcds_all.orient_normals_consistent_tangent_plane(30)
        #o3d.visualization.draw_geometries([pcds_all])
        #o3d.visualization.draw_geometries([pcd_tmp])

        #o3d.io.write_point_cloud(dpath+"/"+suffix+".ply", pcds_all)
    end = time.time()
    print("Duration was:", end-start, "s")

In [ ]:
def process_file_list_comp(file_id, dpath,tform):
    
    # Format filenames
    dpath = dpath.rstrip('/')

    # Split by the directory separator and get the last part
    suffix = os.path.basename(dpath)
    
    file_suffix = suffix + "_nano_" + str(file_id)
    # Construct full file paths using os.path.join for better cross-platform compatibility
    depth_file = os.path.join(dpath, file_suffix + "_depth.png")
    color_file = os.path.join(dpath, file_suffix + ".jpg")
    intrinsics_file = os.path.join(dpath, file_suffix + "_intrinsics.txt")
    # Read the files
    depth_orig = o3d.io.read_image(depth_file)
    device = o3d.core.Device('CUDA:0')
    depth_cuda = o3d.t.io.read_image(depth_file).to(device)
    depth_filtered = depth_cuda.filter_b
    color = o3d.io.read_image(color_file)
   
    

In [ ]:
o3d.visualization.draw_geometries([pcds_tsdf_13[0]])

In [ ]:
#dpath = dpath.rstrip('/')

# Split by the directory separator and get the last part
#suffix = os.path.basename(dpath)

#file_suffix = suffix + "_nano_" + str(11)
# Construct full file paths using os.path.join for better cross-platform compatibility
#depth_file = os.path.join(dpath, file_suffix + "_depth.png")
#color_file = os.path.join(dpath, file_suffix + ".jpg")
#intrinsics_file = os.path.join(dpath, file_suffix + "_intrinsics.txt")
K = np.loadtxt(intrinsics_file)


camera_intrinsics = o3d.camera.PinholeCameraIntrinsic()
camera_intrinsics.set_intrinsics(int(K[0]), int(K[1]), K[2], K[3], K[4], K[5])

cpu = True

# Read the files
depth_orig = o3d.io.read_image(depth_file)
device = o3d.core.Device('CUDA:0')
depth_cuda = o3d.t.io.read_image(depth_file).to(device)
depth_filtered = depth_cuda.filter_bilateral(7,0.004,0.004)
#color = o3d.io.read_image(color_file)
dp_cpu = depth_filtered.cpu().to_legacy()

plt.subplot(1, 2, 1)
plt.title('orig grayscale image')
plt.imshow(depth_orig)
plt.subplot(1, 2, 2)
plt.title('filtered depth image')
plt.imshow(dp_cpu)
plt.show()
p_orig = o3d.geometry.PointCloud.create_from_depth_image(depth_orig,camera_intrinsics)
p_filtered = o3d.geometry.PointCloud.create_from_depth_image(dp_cpu,camera_intrinsics)

#o3d.visualization.draw_geometries([p_orig])

o3d.visualization.draw_geometries([p_filtered])

In [ ]:
max_d = np.max(np.asarray(depth_orig))
print(max_d)

In [ ]:
a13 = '/home/vigir3d/Datasets/cattle_scans/Cattle_11_17_22/Animal_13_6/'
files = glob.glob(a13+"*.ply")
files.sort()
p1 = o3d.io.read_point_cloud(files[0])

In [ ]:
pcd1 = o3d.io.read_point_cloud("/home/vigir3d/Desktop/animal_482_2_demo_cleaned.ply")

In [ ]:
pcdb1_down = pcd1.voxel_down_sample(voxel_size=0.05)
pcdb1_down.orient_normals_consistent_tangent_plane(30)
o3d.io.write_point_cloud("/home/vigir3d/Desktop/animal_482_2_demo_downsampled.ply",pcdb1_down)

In [ ]:
p0 = o3d.io.read_point_cloud(files[0])
p1 = o3d.io.read_point_cloud(files[1])

p2 = o3d.io.read_point_cloud(files[2])
p3 = o3d.io.read_point_cloud(files[3])

p4 = o3d.io.read_point_cloud(files[4])
p5 = o3d.io.read_point_cloud(files[5])

In [ ]:
draw_registration_result(p1,p2,e01)

In [ ]:
draw_registration_result(pcds_tsdf_13[0], pcds_tsdf_13[1],e01)

In [ ]:
a12,e12 = register_two_views_teaser(p1,p2,0.05)

In [ ]:
a01,b01 = register_two_views_teaser(pcds_tsdf_5[0],pcds_tsdf_5[1],0.025) # 0.069


In [ ]:
a01,b01 = register_two_views_teaser(pcds_tsdf_5[0],pcds_tsdf_5[1],vs) # 0.069
a12,b12 = register_two_views_teaser(pcds_tsdf_5[1],pcds_tsdf_5[2],vs) #0.046 | *0.026
at43,b43 = register_two_views_teaser(pcds_tsdf_5[4],pcds_tsdf_5[3],vs) # *0.028 |  0.06 | 0.084
#0.86
a52,b52 = register_two_views_teaser(pcds_tsdf_5[5],pcds_tsdf_5[2],vs) # *0.05 | 0.044000000000000025 | 0.084
a32,b32 = register_two_views_teaser(pcds_tsdf_5[3],pcds_tsdf_5[2],vs) # *0.031  0.084


In [ ]:
for vs in np.arange(0.02,0.03,0.005)  : #vs = 0.042
    f01,d01 = register_two_views_teaser(pcds_tsdf_2[0],pcds_tsdf_2[1],vs) # 0.069
#f12,d12 = register_two_views_teaser(pcds_tsdf_2[1],pcds_tsdf_2[2],vs) #0.046 | *0.026
#ft43,d43 = register_two_views_teaser(pcds_tsdf_2[4],pcds_tsdf_2[3],vs) # *0.028 |  0.06 | 0.084
#0.86

#f52,d52 = register_two_views_teaser(pcds_tsdf_2[5],pcds_tsdf_2[2],vs) # *0.05 | 0.044000000000000025 | 0.084
#f32,d32 = register_two_views_teaser(pcds_tsdf_2[3],pcds_tsdf_2[2],vs) # *0.031  0.084


In [ ]:
h0 = d12 @ d01
h1 = d12
h3 = d32
h4 = d32@d43
h5 = d52


p0 = copy.deepcopy(pcds_tsdf_2[0]).transform(h0)
p1 = copy.deepcopy(pcds_tsdf_2[1]).transform(h1)
p3 = copy.deepcopy(pcds_tsdf_2[3]).transform(h3)
p4 = copy.deepcopy(pcds_tsdf_2[4]).transform(h4)
p5 = copy.deepcopy(pcds_tsdf_2[5]).transform(h5)



In [ ]:
plist = [pcds_tsdf_2[2],p0,p1,p3,p4,p5]
pcd_all_comb_2 = o3d.geometry.PointCloud()
for i in range(len(plist)):
    print("current Addition: ", i )
    pcd_all_comb_2+=plist[i]
    o3d.visualization.draw_geometries([pcd_all_comb_2])

In [ ]:
#for vs in np.arange(0.02,0.087,0.001):
vs = 0.084

#print(vs)
#for vs in np.arange(0.01, 0.086,0.001):
#t01,r01 = register_two_views_teaser(pcds_tsdf[0],pcds_tsdf[1],vs)
#t12,r12 = register_two_views_teaser(pcds_tsdf[1],pcds_tsdf[2],vs) #0.046
#t43,r43 = register_two_views_teaser(pcds_tsdf[4],pcds_tsdf[3],vs) # 0.06
#0.86
#t52,r52 = register_two_views_teaser(pcds_tsdf[5],pcds_tsdf[2],vs) #0.044000000000000025
#t32,r32 = register_two_views_teaser(pcds_tsdf[3],pcds_tsdf[2],vs) # 0.084

# sticth the others together and then add 0
h0 = r12 @ r01
h1 = r12
h3 = r32
h4 = r32@r43
h5 = r52


p0 = copy.deepcopy(pcds_tsdf[0]).transform(h0)

p1 = copy.deepcopy(pcds_tsdf[1]).transform(h1)

p3 = copy.deepcopy(pcds_tsdf[3]).transform(h3)
p4 = copy.deepcopy(pcds_tsdf[4]).transform(h4)
p5 = copy.deepcopy(pcds_tsdf[5]).transform(h5)

o3d.visualization.draw_geometries([p0, p1,p3,p4,p5,pcds_tsdf[2]])


In [ ]:
o3d.visualization.draw_geometries([p1,p3,p4,p5])

In [ ]:
t02 = demo_manual_registration(pcomb,pcds_tsdf[0])

In [ ]:
for vs in np.arange(0.01,0.05,0.01):
    print(vs)
    t02,r02 = register_two_views_teaser(pcds_tsdf[1],pcds_tsdf[0], vs)

In [ ]:
for vs in np.arange(0.02, 0.08,0.01):
    print("*****VS ::: ********* : ", vs)
    t02,r02 = register_two_views_teaser(pcomb_down,pcds_tsdf[0], vs)

In [ ]:
o3d.visualization.draw_geometries([pcomb])

In [ ]:
o3d.visualization.draw_geometries([p1,p3,p4, p5,pcds_tsdf[2]])
pcomb = o3d.geometry.PointCloud()
pcomb = p1+p3+p4+p5+pcds_tsdf[2]

In [ ]:
dpath = '/home/vigir3d/Datasets/cattle_scans/farm_07_28/Animal_cyl1/'
pcds_tsdf_cyl = [process_file_list_comp(file_id, dpath,tform) for file_id,tform in zip(file_ids,tforms)]

In [ ]:
o3d.visualization.draw_geometries(pcds_tsdf_cyl)

In [ ]:
dpath = '/home/vigir3d/Datasets/cattle_scans/farm_07_28/Animal_b1/'
dpath = dpath.rstrip('/')

# Split by the directory separator and get the last part
suffix = os.path.basename(dpath)
print(suffix)

In [ ]:
def colored_ICP_N(source, target):
    
    voxel_radius = [0.04, 0.02, 0.01]
    max_iter = [50, 30, 14]
    current_transformation = np.identity(4)
    #print("3. Colored point cloud registration")
    for scale in range(3):
        iters = max_iter[scale]
        radius = voxel_radius[scale]
        #print("iteration: ", iters, radius, scale)

        #print("3-1. Downsample with a voxel size %.2f" % radius)
        source_down = copy.deepcopy(source).voxel_down_sample(radius)
        target_down = copy.deepcopy(target).voxel_down_sample(radius)

        #print("3-2. Estimate normal.")
        source_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))
        target_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))

        #print("3-3. Applying colored point cloud registration")
        result_icp = o3d.pipelines.registration.registration_colored_icp(
            source_down, target_down, radius, current_transformation,
            o3d.pipelines.registration.TransformationEstimationForColoredICP(),
            o3d.pipelines.registration.ICPConvergenceCriteria(relative_fitness=1e-6,
                                                              relative_rmse=1e-6,
                                                              max_iteration=iters))
        current_transformation = result_icp.transformation
    
   
        #draw_registration_result(source, target, current_transformation)
    return current_transformation


In [ ]:
t01 = colored_ICP(pcds_tsdf[1],pcds_tsdf[2])
draw_registration_result(pcds_tsdf[1],pcds_tsdf[2],t01)

In [ ]:
ptsdf_all = o3d.geometry.PointCloud()
ptsdf_all = pcds_tsdf[0]+pcds_tsdf[1]+pcds_tsdf[2]+pcds_tsdf[3]+pcds_tsdf[4]+pcds_tsdf[5]

In [ ]:
o3d.visualization.draw_geometries(pcds_tsdf_cyl)

In [ ]:
draw_registration_result(ptsdf_all,pcd_all,np.eye(4))

In [ ]:
h0 = t12@t01
h1 = t01
h3 = t32
h4 = t52@t45
h5 = t52

pc0 = copy.deepcopy(pcds_tsdf[0]).transform(h0)
pc1 = copy.deepcopy(pcds_tsdf[1]).transform(h1)
pc3 = copy.deepcopy(pcds_tsdf[3]).transform(h3)
pc4 = copy.deepcopy(pcds_tsdf[4]).transform(h4)
pc5 = copy.deepcopy(pcds_tsdf[5]).transform(h5)

o3d.visualization.draw_geometries([pc0,pc1,pcds_tsdf[2],pc4,pc5])

In [ ]:
t43 = demo_manual_registration(pcds_tsdf[4],pcds_tsdf[3])

In [ ]:
#draw_registration_result(pcds_tsdf_cyl[4],pcds_tsdf_cyl[3],t43)
#draw_registration_result(pcds_tsdf_cyl[0],pcds_tsdf_cyl[1],t01)
draw_registration_result(pcds_tsdf_cyl[1],pcds_tsdf_cyl[2],t12)

In [ ]:
t12 = demo_manual_registration(pcds_tsdf[1],pcds_tsdf[2])

In [ ]:
boxes = pcds_tsdf

In [ ]:
def colored_ICP(source, target):
    #draw_registration_result(source, target, np.eye(4))
    voxel_radius = [0.04, 0.02, 0.01]
    max_iter = [50, 30, 14]
    current_transformation = np.identity(4)
    print("3. Colored point cloud registration")
    for scale in range(3):
        iters = max_iter[scale]
        radius = voxel_radius[scale]
        print("iteration: ", iters, radius, scale)

        print("3-1. Downsample with a voxel size %.2f" % radius)
        source_down = copy.deepcopy(source).voxel_down_sample(radius)
        target_down = copy.deepcopy(target).voxel_down_sample(radius)

        print("3-2. Estimate normal.")
        source_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))
        target_down.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius * 2, max_nn=30))

        print("3-3. Applying colored point cloud registration")
        result_icp = o3d.pipelines.registration.registration_colored_icp(
            source_down, target_down, radius, current_transformation,
            o3d.pipelines.registration.TransformationEstimationForColoredICP(),
            o3d.pipelines.registration.ICPConvergenceCriteria(relative_fitness=1e-6,
                                                              relative_rmse=1e-6,
                                                              max_iteration=iters))
        current_transformation = result_icp.transformation
    
    return current_transformation

In [ ]:
o3d.visualization.draw_geometries([boxes[5],boxes[2]])

In [ ]:
t52 = test_ICP(boxes[5],boxes[2])
draw_registration_result(boxes[5],boxes[2],t52)

In [ ]:
h02 = t12@t01
h12 = t12
h32 = t32
h42 = t32@t43
h52 = t52

p0 = copy.deepcopy(boxes[0]).transform(h02)
p1 = copy.deepcopy(boxes[1]).transform(h12)
p3 = copy.deepcopy(boxes[3]).transform(h32)
p4 = copy.deepcopy(boxes[4]).transform(h42)
p5 = copy.deepcopy(boxes[5]).transform(h52)

In [ ]:

c0 = copy.deepcopy(pcds_tsdf_cyl[0]).transform(h02)
c1 = copy.deepcopy(pcds_tsdf_cyl[1]).transform(h12)
c3 = copy.deepcopy(pcds_tsdf_cyl[3]).transform(h32)
c4 = copy.deepcopy(pcds_tsdf_cyl[4]).transform(h42)
c5 = copy.deepcopy(pcds_tsdf_cyl[5]).transform(h52)
o3d.visualization.draw_geometries([c0,c1,c3,c4,c5])
#o3d.visualization.draw_geometries(pcds_tsdf_cyl)

In [ ]:
pcd_no_5 = o3d.geometry.PointCloud()
pcd_no_5 = p0+p1+p3+p4+boxes[2]

r52 = register_two_views_teaser(pcd_no_5,boxes[5],0.045)

In [ ]:
o3d.visualization.draw_geometries([p0,p1,p3,p4,boxes[2]])

In [ ]:
for vs in np.arange(0.04, 0.07,0.001):
    r50, t50 = register_two_views_teaser(boxes[5],boxes[3],vs)

In [ ]:
# for farm_07_28 Animal_c1
#t01 = colored_ICP(boxes[0],boxes[1])
#draw_registration_result(boxes[0],boxes[1],t01)
#t12 = colored_ICP(boxes[1],boxes[2])
#draw_registration_result(boxes[1],boxes[2],t12)
#r32, t32 = register_two_views_teaser(boxes[3],boxes[2],vs) # 0.047
#draw_registration_result(boxes[3],boxes[2],t32)
#r43, t43 = register_two_views_teaser(boxes[4],boxes[3],vs) # 0.025
#draw_registration_result(boxes[4],boxes[3],t43)
#r52, t52 = test_ICP(boxes[5],boxes[4])
#draw_registration_result(boxes[5],boxes[2],t52)

In [ ]:
tpath = '/home/vigir3d/Software/programs/Cattle_Scanner/'
h02 = np.loadtxt(tpath+'c_icp_0_2.txt')
h12 = np.loadtxt(tpath+ 'c_icp_1_2.txt')
h32 = np.loadtxt(tpath+ 'c_icp_3_2.txt')
h42 = np.loadtxt(tpath+ 'c_icp_4_2.txt')
h52 = np.loadtxt(tpath+ 'c_icp_5_2.txt')

In [ ]:
p0 = copy.deepcopy(pcds_tsdf[0]).transform(h02)
p1 = copy.deepcopy(pcds_tsdf[1]).transform(h12)
p3 = copy.deepcopy(pcds_tsdf[3]).transform(h32)
p4 = copy.deepcopy(pcds_tsdf[4]).transform(h42)
p5 = copy.deepcopy(pcds_tsdf[5]).transform(h52)

In [ ]:
o3d.visualization.draw_geometries([p0,p1,p3,p4,p5])

In [ ]:
t52 = demo_manual_registration(pcds_tsdf[5],pcds_tsdf[2])
#r52, t52 = register_two_views_teaser(pcds_tsdf[5],pcds_tsdf[2],vs) #0.03900000000000002


In [ ]:
#for vs in np.arange(0.06,0.09,0.005) : #vs = 0.035
 #   print(vs)
#for vs in np.arange(0.02,0.08,0.001): #vs = 0.095
#   print("Current VS : ", vs)
#r01, t01 = register_two_views_teaser(pcds_tsdf[0],pcds_tsdf[1],vs)# 0.035
#r12, t12 = register_two_views_teaser(pcds_tsdf[1],pcds_tsdf[2],vs) # 0.065
#r32, t32 = register_two_views_teaser(pcds_tsdf[3],pcds_tsdf[2],vs) # 0.075
#r45, t45 = register_two_views_teaser(pcds_tsdf[4],pcds_tsdf[5],vs)
vs =0.039
r52, t52 = register_two_views_teaser(pcds_tsdf[5],pcds_tsdf[2],vs) #0.03900000000000002


In [ ]:
t01 = test_ICP(pcds_tsdf[0],pcds_tsdf[1])
t12 = test_ICP(pcds_tsdf[1],pcds_tsdf[2])
t32 = test_ICP(pcds_tsdf[3],pcds_tsdf[2])
t43 = test_ICP(pcds_tsdf[4],pcds_tsdf[3])
t52 = test_ICP(pcds_tsdf[5],pcds_tsdf[2])

h0 = t12@t01
h1 = t12
h3 = t32
h4 = t32@t43
h5 = t52

p0 = copy.deepcopy(pcds_tsdf[0]).transform(h0)
p1 = copy.deepcopy(pcds_tsdf[1]).transform(h1)
p3 = copy.deepcopy(pcds_tsdf[3]).transform(h3)
p4 = copy.deepcopy(pcds_tsdf[4]).transform(h4)
p5 = copy.deepcopy(pcds_tsdf[5]).transform(h5)

pcd_all = o3d.geometry.PointCloud()
pcd_all = p0+p1+pcds_tsdf[2]+p3+p4+p5

In [ ]:
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])


def pick_points(pcd):
    print("")
    print(
        "1) Please pick at least three correspondences using [shift + left click]"
    )
    print("   Press [shift + right click] to undo point picking")
    print("2) After picking points, press 'Q' to close the window")
    vis = o3d.visualization.VisualizerWithEditing()
    vis.create_window()
    vis.add_geometry(pcd)
    vis.run()  # user picks points
    vis.destroy_window()
    print("")
    return vis.get_picked_points()


def demo_manual_registration(src, tgt):
    print("Demo for manual ICP")
    source = src
    target = tgt
    print("Visualization of two point clouds before manual alignment")
    draw_registration_result(source, target, np.identity(4))

    # pick points from two point clouds and builds correspondences
    picked_id_source = pick_points(source)
    picked_id_target = pick_points(target)
    assert (len(picked_id_source) >= 3 and len(picked_id_target) >= 3)
    assert (len(picked_id_source) == len(picked_id_target))
    corr = np.zeros((len(picked_id_source), 2))
    corr[:, 0] = picked_id_source
    corr[:, 1] = picked_id_target

    # estimate rough transformation using correspondences
    print("Compute a rough transform using the correspondences given by user")
    p2p = o3d.pipelines.registration.TransformationEstimationPointToPoint()
    trans_init = p2p.compute_transformation(source, target,
                                            o3d.utility.Vector2iVector(corr))

    # point-to-point ICP for refinement
    print("Perform point-to-point ICP refinement")
    threshold = 0.03  # 3cm distance threshold
    reg_p2p = o3d.pipelines.registration.registration_icp(
        source, target, threshold, trans_init,
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    draw_registration_result(source, target, reg_p2p.transformation)
    return reg_p2p.transformation
    print("")

In [ ]:
o3d.visualization.draw_geometries([pcd_all])

In [ ]:

def test_ICP(src,tgt):
    threshold = 0.005
    #print("Apply point-to-point ICP")
    reg_p2p = o3d.pipelines.registration.registration_icp(
        src, tgt, threshold, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    #print(reg_p2p)
    #print("Transformation is:")
    #print(reg_p2p.transformation)
    
    #draw_registration_result(src, tgt, reg_p2p.transformation)
    return reg_p2p.transformation


In [ ]:
t01 = colored_ICP(pcds_tsdf[0],pcds_tsdf[1])

In [ ]:
draw_registration_result(pcds_tsdf[0],pcds_tsdf[1],np.eye(4))

In [ ]:
o3d.visualization.draw_geometries([pcds_tsdf[0]])

In [ ]:
t12,r12 = register_two_views_teaser(pcds_tsdf[1],pcds_tsdf[2],0.03)

In [ ]:
t32,r32 = register_two_views_teaser(pcds_tsdf[4],pcds_tsdf[5],0.05)

In [34]:
cv2.imwrite()

error: OpenCV(4.8.0) :-1: error: (-5:Bad argument) in function 'imwrite'
> Overload resolution failed:
>  - imwrite() missing required argument 'filename' (pos 1)
>  - imwrite() missing required argument 'filename' (pos 1)


In [62]:
def segment_images_paper(depth_image_array, rgb_image_array,datpath):
    #depth_image_array = np.asarray(last_frame.depth)
    print("Pre:",np.max(depth_image_array), " -- ", np.min(depth_image_array))
    depth_PIL = Image.fromarray(depth_image_array).convert("RGB")
    rgb_PIL = Image.fromarray(rgb_image_array)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    rgb_mask = get_model_mask(model_rgb, rgb_PIL)
    depth_mask = get_model_mask(model_depth, depth_PIL)
   
    rgb_mask_np = rgb_mask.cpu().numpy().squeeze(0)
    depth_mask_np = depth_mask.cpu().numpy().squeeze(0)

    rgb_mask_np = (rgb_mask_np * 255).astype(np.uint8)
    depth_mask_np = (depth_mask_np * 255).astype(np.uint8)

    cv2.imwrite(datpath+"rgb_mask.png", rgb_mask_np)
    cv2.imwrite(datpath+"depth_mask.png", depth_mask_np)

    # ... rest of your code ...

    # Check if depth_mask is empty, if so use rgb_mask
    if depth_mask.nelement() == 0:
        mask_combined = rgb_mask
    else:
        mask_combined = depth_mask | rgb_mask  # 1 vote arbitration (OR the masks)

    # Convert tensor to numpy array and ensure the right datatype
    mask_combined_np = mask_combined.cpu().numpy().squeeze(0)  # Convert to NumPy and remove extra dimensions
    mask_combined_np = (mask_combined_np * 255).astype(np.uint8)  # Scale to 8-bit if necessary

    # Save the combined mask
    cv2.imwrite(datpath + "combined_mask.png", mask_combined_np)

    # Process RGB image
    mask_image = mask_combined_np[..., None]  # Add a channel dimension if necessary
    fg_image_rgb = rgb_image_array * mask_image
    fg_image_rgb_bgr = cv2.cvtColor(fg_image_rgb, cv2.COLOR_RGB2BGR)  # Convert to BGR for OpenCV

    # Save the RGB image
    cv2.imwrite(datpath + "fg_color.png", fg_image_rgb_bgr)

    # Process Depth image
    squeezed_mask = np.squeeze(mask_image)  # Remove channel dimension if necessary
    fg_image_depth = (depth_image_array * squeezed_mask) * 1000  # Upscale

    # Save the depth image
    cv2.imwrite(datpath + "fg_depth.png", fg_image_depth)

  

In [37]:
datpath = '/home/vigir3d/Datasets/cattle_scans/farm_scan1/Animal_482_2/anima_482_12/'

In [43]:
color = o3d.io.read_image(datpath+'/color/00011.jpg')
depth = o3d.io.read_image(datpath+'/depth/00011.png')

In [45]:
c = np.asarray(color)
d = np.asarray(depth)


In [63]:
segment_images_paper(d,c,datpath)

Pre: 5885  --  0
